# Root finding for Kepler's equation

A PROTOTYPE of the compute gate (Prompt 268, COMPUTE_GATE_II.md): it is not in the catalogue and not shipped. It reads its input as the text `gravitas_input` and its settings as the JSON text `gravitas_params`, and returns `gravitas_output`, a dict of numbers. Python only, numpy and scipy only.


In [ ]:
import json, math
import numpy as np
doc = json.loads(gravitas_input)
assert doc.get("format") == "gravitas.orbital-system" and doc.get("version") == 1, "this notebook reads gravitas.orbital-system/1"
es = [float(b["e"]) for b in doc["bodies"][1:] if "e" in b]
assert all(0.0 <= e < 1.0 for e in es), "refused: Kepler's equation as written needs 0 <= e < 1"
es = es + [0.5, 0.9, 0.99, 0.999]
Ms = [2 * math.pi * k / 720 for k in range(721)] + [1e-6, 1e-3, 2 * math.pi - 1e-3]
TOL = 1e-14

In [ ]:
def newton_history(M, e, steps):
    E, hist = M, []
    for _ in range(steps):
        E = E - (E - e * math.sin(E) - M) / (1 - e * math.cos(E))
        hist.append(E)
    return hist

def newton(M, e):
    E = M
    for k in range(1, 101):
        dE = (E - e * math.sin(E) - M) / (1 - e * math.cos(E))
        E -= dE
        if abs(dE) <= TOL:
            return E, k, True
    return E, 100, False

def halley(M, e):
    E = M
    for k in range(1, 101):
        f = E - e * math.sin(E) - M
        f1 = 1 - e * math.cos(E)
        f2 = e * math.sin(E)
        dE = f / (f1 - f * f2 / (2 * f1))
        E -= dE
        if abs(dE) <= TOL:
            return E, k, True
    return E, 100, False

def bisect(M, e):
    lo, hi = M - e, M + e
    for k in range(1, 101):
        mid = 0.5 * (lo + hi)
        if mid - e * math.sin(mid) - M > 0:
            hi = mid
        else:
            lo = mid
        if hi - lo <= TOL:
            return 0.5 * (lo + hi), k, True
    return 0.5 * (lo + hi), 100, False

roots, stats = {}, {}
for e in es:
    runs = {name: [fn(M, e) for M in Ms] for name, fn in (("newton", newton), ("halley", halley), ("bisection", bisect))}
    roots[repr(e)] = [r[0] for r in runs["bisection"]]
    stats[repr(e)] = {name: {"failures": sum(1 for r in rr if not r[2]), "meanIterations": float(np.mean([r[1] for r in rr if r[2]])) if any(r[2] for r in rr) else None, "maxIterations": max(r[1] for r in rr)} for name, rr in runs.items()}
orders = []
for M, root in zip(Ms, roots[repr(0.5)]):
    errs = [abs(E - root) for E in newton_history(M, 0.5, 8)]
    for k in range(len(errs) - 2):
        if errs[k] < 1e-2 and errs[k + 2] > 1e-14 and errs[k + 1] > 0 and errs[k] > 0:
            orders.append(math.log(errs[k + 2] / errs[k + 1]) / math.log(errs[k + 1] / errs[k]))
            break
assert orders, "no mesh point gave an error sequence to measure the order from"
gravitas_output = {"answer": {"e": es, "M": Ms, "bisectionRoots": roots, "stats": stats, "newtonOrder": float(np.median(orders)), "orderSamples": len(orders)}}